## 依赖导入

本例集合覆盖插值与拟合两大类的数值实现：一维插值（Vandermonde 行列式与拉格朗日多项式、分段线性、三次样条、UnivariateSpline 积分）、二维插值（interp2d 网格插值、griddata 散点插值）、曲线拟合（最小二乘公式、伪逆、curve_fit、least_squares、leastsq）以及用 cvxpy 求解带约束的最小二乘。统一导入 cvxpy、matplotlib、numpy 与 scipy.interpolate、scipy.optimize 的相关函数。

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
from numpy.linalg import norm
from scipy.interpolate import UnivariateSpline, griddata, interp1d, interp2d, lagrange
from scipy.optimize import curve_fit, least_squares, leastsq

## 全局绘图配置

设置全局 `plt.rc`：启用 TeX 渲染数学公式，中文字体 SimHei 用于正常显示中文标签，字号先设 15 再设为 16，并关闭坐标轴负号的 Unicode 显示，保证后续各图的中文标题与公式标签正常显示。

In [ ]:
plt.rc("text", usetex=True)
plt.rc("font", size=15)
plt.rc("font", family="SimHei")
plt.rc("axes", unicode_minus=False)
plt.rc("font", size=16)

## 5.1  Vandermonde 矩阵的多项式插值

给定 6 个数据点 $x=1,\dots,6$ 与对应值 [16,18,21,17,15,12]：用 `np.vander(x0)` 构造 Vandermonde 矩阵，`np.linalg.inv(A) @ y0` 解出插值多项式从高次幂到低次幂的系数 p（即解 $Ap=y_0$），打印 4 位小数；`np.polyval(p, [1.5, 2.6])` 计算多项式在两点处的预测函数值。图中以 `'o'` 散点标出已知数据点，并在 `np.linspace(1, 6, 100)` 的加密节点上叠加插值曲线（源程序 ex7_1.py）。

In [ ]:
x0 = np.arange(1, 7)
y0 = np.array([16, 18, 21, 17, 15, 12])
A = np.vander(x0)
p = np.linalg.inv(A) @ y0
print("从高次幂到低次幂的系数为：", np.round(p, 4))
yh = np.polyval(p, [1.5, 2.6])
print("预测值为：", np.round(yh, 4))
fig, ax = plt.subplots()
ax.plot(x0, y0, "o")
xt = np.linspace(1, 6, 100)
ax.plot(xt, np.polyval(p, xt))
plt.show()

## 5.2  拉格朗日插值多项式的系数

对同样 6 个数据点，`scipy.interpolate.lagrange(x0, y0)` 直接返回过全点的插值多项式系数（从高次幂到低次幂），打印系数并用 `np.polyval` 计算 $x=1.5,\;2.6$ 处的预测函数值，结果与 5.1 的 Vandermonde 求逆法一致（源程序 ex7_2.py）。

In [ ]:
x0 = np.arange(1, 7)
y0 = np.array([16, 18, 21, 17, 15, 12])
p = lagrange(x0, y0)
print("从高次幂到低次幂的系数为：", np.round(p, 4))
yh = np.polyval(p, [1.5, 2.6])
print("预测值为：", np.round(yh, 4))

## 5.3  等距拉格朗日插值与 Runge 现象

对 Runge 函数 $f(x)=\frac{1}{1+x^2}$，在 $[-5,5]$ 的等距节点 `np.linspace(-5, 5, n+1)` 上分别作 $n=6,8,10$ 的拉格朗日插值：`fun(n)` 返回 $n$ 次插值多项式的系数。循环中每个 $n$ 新建一个画布绘制对应插值曲线（线型取 `s=["--*b", "-.", "-p"]`），循环结束后在最后一个子图的坐标轴上叠加真实曲线 $1/(1+x^2)$，图例依次为 $n=6$、$n=8$、$n=10$ 与真实函数，演示等距高次插值在区间端部附近显著偏离真函数的 Runge 现象（源程序 ex7_3.py）。

In [ ]:
def yx(x):
    return 1 / (1 + x**2)


def fun(n):
    x = np.linspace(-5, 5, n + 1)
    return lagrange(x, yx(x))


x0 = np.linspace(-5, 5, 100)
N = [6, 8, 10]
s = ["--*b", "-.", "-p"]
for k in range(len(N)):
    p = fun(N[k])
    fig, ax = plt.subplots()
    ax.plot(x0, np.polyval(p, x0), s[k])
ax.plot(x0, yx(x0))
ax.legend(["$n=6$", "$n=8$", "$n=10$", "$1/(1+x^2)$"])
plt.show()

## 5.4  一维三种插值方法的比较

读取 `data/7_4.txt`（第 1 行横坐标 x0、第 2 行纵坐标 y0），在 `np.linspace(0, 15, 151)` 上做三种插值：`interp1d(x0, y0)` 分段线性插值、`lagrange(x0, y0)` 多项式插值（用 `np.polyval` 求值）、`interp1d(..., "cubic")` 三次样条插值。对三次样条结果以差分商 `dy/dx` 的首个分量作为 $x=0$ 处斜率的数值解 dyx0；在尾段 `y3[130:]` 中找最小值 ymin 及其对应横坐标列表 xmin，三者均打印。最后 `plt.subplots(1, 3)` 并排三个子图分别绘制三种插值曲线，标题依次为“分段线性插值”“Lagrange 插值”“三次样条插值”（源程序 ex7_4.py）。

In [ ]:
a = np.loadtxt("data/7_4.txt")
x0 = a[0]
y0 = a[1]
x = np.linspace(0, 15, 151)
yx1 = interp1d(x0, y0)
y1 = yx1(x)
p2 = lagrange(x0, y0)
y2 = np.polyval(p2, x)
yx3 = interp1d(x0, y0, "cubic")
y3 = yx3(x)
dx = np.diff(x)
dy = np.diff(y3)
dyx = dy / dx
dyx0 = dyx[0]
xt = x[130:]
yt = y3[130:]
ymin = min(yt)
xmin = [xt[ind] for ind, v in enumerate(yt) if v == ymin]
print("x=0 处斜率的数值解为：", dyx0)
print("xmin=", xmin)
print("ymin=", ymin)
fig, (ax, ax2, ax3) = plt.subplots(1, 3)
ax.plot(x, y1)
ax.set(title="分段线性插值")
ax2.plot(x, y2)
ax2.set(title="Lagrange 插值")
ax3.plot(x, y3)
ax3.set(title="三次样条插值")
plt.show()

## 5.5  三次样条的定积分（两种方法）

给定 4 对数据 (t0, v0)。第 1 种方法：`UnivariateSpline(t0, v0)` 拟合三次样条函数，打印其系数 `get_coeffs()`，并调用自带的 `integral(0.15, 0.18)` 求样条函数在区间上的定积分；第 2 种方法：`interp1d(t0, v0, "cubic")` 得到样条求值函数，在 $[0.15,0.18]$ 的 200 个等距节点上取值后用 `np.trapezoid` 作梯形数值积分。打印两种方法的积分值（源程序 ex7_5.py）。

In [ ]:
t0 = np.linspace(0.15, 0.18, 4)
v0 = np.array([3.5, 1.5, 2.5, 2.8])
sp1 = UnivariateSpline(t0, v0)
print(sp1.get_coeffs())
print("第 1 种方法的积分值：", sp1.integral(0.15, 0.18))
sp2 = interp1d(t0, v0, "cubic")
tn = np.linspace(0.15, 0.18, 200)
vn = sp2(tn)
I2 = np.trapezoid(vn, tn)
print("第 2 种方法的积分值：", I2)

## 5.6  地块边界曲线的周长与面积（方法一：UnivariateSpline）

读取 `data/7_6.txt`，每 3 个数一组：`a[::3]`、`a[1::3]`、`a[2::3]` 分别取出横坐标 x0、下边界纵坐标 y1、上边界纵坐标 y2（展平后以 `*-`、`.-` 描点绘图）。对两个边界分别用 `UnivariateSpline` 作三次样条 f1/f2，`derivative(1)` 求导函数，在 1000 个密集节点 x 上对弧长元素 $\sqrt{1+y'^2}$（两边界相加）与高度差 $f_2-f_1$ 分别作 `np.trapezoid` 积分得周长与面积；按水平方向 18 对应实际 40 换算（周长乘 $40/18$、面积乘 $1600/18^2$），打印周长 L、面积 S（round 4 位）及 S 相对 41288 的相对误差 delta（源程序 ex7_6_1.py）。

In [ ]:
a = np.loadtxt("data/7_6.txt")
x0 = a[::3].flatten()
y1 = a[1::3].flatten()
y2 = a[2::3].flatten()
fig, ax = plt.subplots()
ax.plot(x0, y1, "*-")
ax.plot(x0, y2, ".-")
f1 = UnivariateSpline(x0, y1)
f2 = UnivariateSpline(x0, y2)
d1 = f1.derivative(1)
d2 = f2.derivative(1)
x = np.linspace(x0[0], x0[-1], 1000)
d10 = d1(x)
d20 = d2(x)
L = np.trapezoid(np.sqrt(1 + d10**2) + np.sqrt(1 + d20**2), x)
L = L / 18 * 40
print("周长 L=", round(L, 4))
S = np.trapezoid(f2(x) - f1(x), x)
S = S / 18**2 * 1600
print("面积 S=", round(S, 4))
delta = (S - 41288) / 41288
print("相对误差 delta=", round(delta, 4))
plt.show()

方法二（源程序 ex7_6_2.py）：同样的数据（x0、下边界 y1、上边界 y2 按每 3 个数一组取出并展平），不构造样条——直接用 `np.gradient(y1, x0)`、`np.gradient(y2, x0)` 以中心差商近似两边界导数，代入弧长元素 $\sqrt{1+y'^2}$ 后对 x0 作 `np.trapezoid` 积分得周长，面积直接对 $y_2-y_1$ 积分；$40/18$、$1600/18^2$ 的换算系数与打印输出同方法一。

In [ ]:
a = np.loadtxt("data/7_6.txt")
x0 = a[::3].flatten()
y1 = a[1::3].flatten()
y2 = a[2::3].flatten()
L = np.trapezoid(
    np.sqrt(1 + np.gradient(y1, x0) ** 2) + np.sqrt(1 + np.gradient(y2, x0) ** 2), x0
)
L = L / 18 * 40
print("周长 L=", round(L, 4))
S = np.trapezoid(y2 - y1, x0)
S = S / 18**2 * 1600
print("面积 S=", round(S, 4))
delta = (S - 41288) / 41288
print("相对误差 delta=", round(delta, 4))

## 5.7  高程网格的双线性插值与曲面面积

`np.loadtxt("data/7_7.txt")` 读入高程矩阵 z（列对应 $x=0,100,\dots,1400$，行对应 $y=1200,1100,\dots,0$），`interp2d(x, y, z)` 作双线性插值，在加密网格 xn=linspace(0,1400,141)、yn=linspace(0,1200,121) 上计算插值点的函数值 zn。对每个网格小方格用对角线 p1–p3 分成两个三角形：`norm`（numpy.linalg）计算三边长，Heron 公式 $\sqrt{L(L-a)(L-b)(L-c)}$ 求两个三角形面积并累加为曲面总面积 s（打印）。左子图 `contour` 等高线并加标签，右子图 `projection="3d"` 用 `plot_surface` 绘制地形曲面（viridis 色图）（源程序 ex7_7.py）。

In [ ]:
z = np.loadtxt("data/7_7.txt")
x = np.arange(0, 1500, 100)
y = np.arange(1200, -100, -100)
f = interp2d(x, y, z)
xn = np.linspace(0, 1400, 141)
yn = np.linspace(0, 1200, 121)
zn = f(xn, yn)
m = len(xn)
n = len(yn)
s = 0
for i in np.arange(m - 1):
    for j in np.arange(n - 1):
        p1 = np.array([xn[i], yn[j], zn[j, i]])
        p2 = np.array([xn[i + 1], yn[j], zn[j, i + 1]])
        p3 = np.array([xn[i + 1], yn[j + 1], zn[j + 1, i + 1]])
        p4 = np.array([xn[i], yn[j + 1], zn[j + 1, i]])
        p12 = norm(p1 - p2)
        p23 = norm(p3 - p2)
        p13 = norm(p3 - p1)
        p14 = norm(p4 - p1)
        p34 = norm(p4 - p3)
        L1 = (p12 + p23 + p13) / 2
        s1 = np.sqrt(L1 * (L1 - p12) * (L1 - p23) * (L1 - p13))
        L2 = (p13 + p14 + p34) / 2
        s2 = np.sqrt(L2 * (L2 - p13) * (L2 - p14) * (L2 - p34))
        s = s + s1 + s2
print("区域的面积为：", s)
fig = plt.figure()
ax2 = fig.add_subplot(1, 2, 1)
contr = ax2.contour(xn, yn, zn)
ax2.clabel(contr)
ax2.set(xlabel="$x$")
ax2.set_ylabel("$y$", rotation=0)
ax = fig.add_subplot(1, 2, 2, projection="3d")
X, Y = np.meshgrid(xn, yn)
ax.plot_surface(X, Y, zn, cmap="viridis")
ax.set(xlabel="$x$", ylabel="$y$", zlabel="$z$")
plt.show()

## 5.8  双三次样条插值求最大高程

读入 `data/7_8.txt` 的 5×4 高程矩阵 z0，对应节点 x0=linspace(100, 500, 5)、y0=linspace(100, 400, 4)，`interp2d(x0, y0, z0, "cubic")` 作双三次样条插值，在加密网格 xn（41 点）、yn（31 点）上求插值函数值 zn。`zn.max()` 得矩阵元素最大值 zm，`np.where(zn == zm)` 定位行列下标，打印对应的 $x=x_n[i_x]$、$y=y_n[i_y]$ 与最大高程（round 4 位）（源程序 ex7_8.py）。

In [ ]:
z0 = np.loadtxt("data/7_8.txt")
x0 = np.linspace(100, 500, 5)
y0 = np.linspace(100, 400, 4)
f = interp2d(x0, y0, z0, "cubic")
xn = np.linspace(100, 500, 41)
yn = np.linspace(100, 400, 31)
zn = f(xn, yn)
zm = zn.max()
iy, ix = np.where(zn == zm)
print("x=", xn[ix])
print("y=", yn[iy])
print("最大高程为：", round(zm, 4))

## 5.9  散点数据的 griddata 插值与曲面绘制

读入 `data/7_9.txt`：第 1 行 x、第 2 行 y、第 3 行取负后的 z。`xy=vstack([x, y]).T` 为散点坐标，在 x、y 各自 min–max 范围的 100×100 `meshgrid` 节点上，`griddata` 分别作 `method="cubic"` 三次样条插值得 zn1、`method="nearest"` 最近邻插值得 zn2，并把 zn1 中的 NaN 用 zn2 对应值替换掉。左子图 `projection="3d"` 绘制 `plot_surface` 曲面（viridis），右子图 `contour` 等高线加标签，图保存为 `data/7_9.png`（源程序 ex7_9.py）。

In [ ]:
a = np.loadtxt("data/7_9.txt")
x = a[0]
y = a[1]
z = -a[2]
xy = np.vstack([x, y]).T
xn = np.linspace(x.min(), x.max(), 100)
yn = np.linspace(y.min(), y.max(), 100)
xng, yng = np.meshgrid(xn, yn)
zn1 = griddata(xy, z, (xng, yng), method="cubic")
zn2 = griddata(xy, z, (xng, yng), method="nearest")
zn1[np.isnan(zn1)] = zn2[np.isnan(zn1)]
fig = plt.figure()
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot_surface(xng, yng, zn1, cmap="viridis")
ax.set(xlabel="$x$", ylabel="$y$", zlabel="$z$")
ax2 = fig.add_subplot(1, 2, 2)
c = ax2.contour(xn, yn, zn1)
ax2.clabel(c)
fig.savefig("data/7_9.png", dpi=500)
plt.show()

## 5.10  线性最小二乘拟合的两种解法

$t=0,\dots,7$（`np.arange(8)`）与降温数据 y。第 1 种：回归公式 $a_1=\sum(t-\bar t)(y-\bar y)/\sum(t-\bar t)^2$、$b_1=\bar y-a_1\bar t$，其中求和用 ndarray 的 `.sum()`；第 2 种：设计矩阵 `A = vstack([t, ones]).T`，`np.linalg.pinv(A) @ y` 伪逆解法向量。分别打印两种方法的拟合系数（源程序 ex7_10.py）。

In [ ]:
t = np.arange(8)
y = np.array([27.0, 26.8, 26.5, 26.3, 26.1, 25.7, 25.3, 24.8])
tb = t.mean()
yb = y.mean()
a1 = ((t - tb) * (y - yb)).sum() / ((t - tb) ** 2).sum()
b1 = yb - a1 * tb
print("拟合的多项式系数：", [a1, b1])
A = np.vstack([t, np.ones(len(t))]).T
p = np.linalg.pinv(A) @ y
print("拟合的多项式系数：", p)

## 5.11  五点拟合二次隐式曲线

读 `data/7_11.txt`（5 个点的 x、y 两行），代入二次方程 $c_0x^2+c_1xy+c_2y^2+c_3x+c_4y+1=0$：按列堆叠 [x², xy, y², x, y] 得 5×5 系数矩阵 A，常数项 b 为 -1 向量，`np.linalg.inv(A) @ b` 解出拟合参数 c（round 4 位打印）。在 x∈[3,8]、y∈[-1,5] 的 `meshgrid` 网格上计算 $f(x,y)=c_0x^2+c_1xy+c_2y^2+c_3x+c_4y$，绘制高度为 -1 的等高线 `contour(..., [-1])`，即过这五点的二次曲线（源程序 ex7_11.py）。

In [ ]:
a = np.loadtxt("data/7_11.txt")
x = a[0]
y = a[1]
A = np.vstack([x**2, x * y, y**2, x, y]).T
b = -np.ones(5)
c = np.linalg.inv(A) @ b
print("拟合的系数为：\n", np.round(c, 4))


def f(x, y):
    return c[0] * x**2 + c[1] * x * y + c[2] * y**2 + c[3] * x + c[4] * y


x = np.linspace(3, 8, 100)
y = np.linspace(-1, 5, 100)
x, y = np.meshgrid(x, y)
z = f(x, y)
fig, ax = plt.subplots()
ax.contour(x, y, z, [-1])
ax.set(xlabel="$x$")
ax.set_ylabel("$y$", rotation=0)
plt.show()

## 5.12  cvxpy 约束最小二乘拟合（指数加对数模型）

读 `data/7_12.txt`（第 1 行 x、第 2 行 y），拟合 $y=ae^{x}+b\ln x$：决策变量 `t = cp.Variable(2, pos=True)`（t[0]=a、t[1]=b，带正性约束），设计矩阵 `c = vstack([exp(x0), log(x0)]).T`，目标 `cp.Minimize(cp.sum_squares(c @ t - y0))`，约束 `sum(t) <= 1`（此处 sum 作用于 cvxpy 表达式，保持原写法），用 CVXOPT 求解并打印最优解 t.value（源程序 ex7_12.py）。

In [ ]:
a = np.loadtxt("data/7_12.txt")
x0 = a[0]
y0 = a[1]
t = cp.Variable(2, pos=True)
c = np.vstack([np.exp(x0), np.log(x0)]).T
obj = cp.Minimize(cp.sum_squares(c @ t - y0))
con = [sum(t) <= 1]
prob = cp.Problem(obj, con)
prob.solve(solver="CVXOPT")
print("最优解为：\n", t.value)

## 5.13  指数模型线性化（取对数后 polyfit）

读 `data/7_13.txt` 第 1 行 t、第 2 行 y。对模型 $y=ke^{mt}$ 两边取对数化为线性关系，`np.polyfit(t, np.log(y), 1)` 拟合一次多项式：斜率 p[0] 即 m、$k=e^{p[1]}$，分别打印多项式系数与 round 4 位后的 m、k（源程序 ex7_13.py）。

In [ ]:
a = np.loadtxt("data/7_13.txt")
t = a[0]
y = np.log(a[1])
p = np.polyfit(t, y, 1)
print("多项式的系数为：", p)
print("m=", round(p[0], 4))
print("k=", round(np.exp(p[1]), 4))

## 5.14  curve_fit 直接非线性拟合指数模型

同一组数据（`data/7_13.txt` 的 t0、y0），写模型函数 $y(t;k,m)=ke^{mt}$，`curve_fit(y, t0, y0)` 直接做非线性最小二乘（与 5.13 的线性化途径互为对照），打印 round 4 位的拟合参数 p，并用拟合结果预测 t=5、8 处的函数值（源程序 ex7_14.py）。

In [ ]:
a = np.loadtxt("data/7_13.txt")
t0 = a[0]
y0 = a[1]


def y(t, k, m):
    return k * np.exp(m * t)


p, pcov = curve_fit(y, t0, y0)
print("拟合的参数为：", np.round(p, 4))
yh = y(np.array([5, 8]), *p)
print("预测值为：", np.round(yh, 4))

## 5.15  curve_fit 拟合双变量模型

给定自变量两行组成的 xy0（2×8）与观测向量 z0（长 8），模型 $z=a e^{t_0}+c t_1^{2}$（函数 z(t, a, b, c) 中 t[0]、t[1] 分别是两个自变量分量），`curve_fit(z, xy0, z0)` 后打印 a、b、c 的拟合值（源程序 ex7_15.py）。

In [ ]:
xy0 = np.array([[6, 2, 6, 7, 4, 2, 5, 9], [4, 9, 5, 3, 8, 5, 8, 2]])
z0 = np.array([5, 2, 1, 9, 7, 4, 3, 3])


def z(t, a, b, c):
    return a * np.exp(b * t[0]) + c * t[1] ** 2


p, pcov = curve_fit(z, xy0, z0)
print("a，b，c 的拟合值为：", p)

## 5.16  二维高斯曲面拟合（curve_fit / least_squares / leastsq 三种方法）

`np.random.seed(1)` 保证随机数可一致比较。在 x∈[-6,6]（m=200）、y∈[-8,8]（n=300）的 `meshgrid` 上取无噪声函数值 $z=e^{-((x-m_1)^2+(y-m_2)^2)/(2s^2)}$（真参数 m1=1、m2=4、s=6），展平为散点 xy 并加 0.2 倍标准正态噪声得 zr。方法一：`curve_fit(zxy, xy, zr)` 直接拟合三个参数并打印，`zxy(xy, *p1)` 求拟合函数值、reshape 回网格后用 `projection="3d"` 的 `plot_surface` 绘制（gist_rainbow 色图），图保存为 `data/7_15.png`。方法二、三：以 `p0 = np.random.randn(3)` 为初值，对误差向量 `err = 模型 - z` 分别用 `least_squares`、`leastsq` 迭代并打印结果 p2、p3（源程序 ex7_16.py）。

In [ ]:
np.random.seed(1)  # noqa: NPY002
m = 200
n = 300
x = np.linspace(-6, 6, m)
y = np.linspace(-8, 8, n)
x2, y2 = np.meshgrid(x, y)
x3 = x2.flatten()
y3 = y2.flatten()
xy = np.vstack([x3, y3])


def zxy(t, m1, m2, s):
    return np.exp(-((t[0] - m1) ** 2 + (t[1] - m2) ** 2) / (2 * s**2))


z = zxy(xy, 1, 4, 6)
zr = z + 0.2 * np.random.normal(size=z.shape)  # noqa: NPY002
p1 = curve_fit(zxy, xy, zr)[0]
print("三个参数的拟合值分别为：", p1)
zn = zxy(xy, *p1)
zn2 = np.reshape(zn, x2.shape)
fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot_surface(x2, y2, zn2, cmap="gist_rainbow")

p0 = np.random.randn(3)  # noqa: NPY002


def fun(t, x, y):
    return np.exp(-((x - t[0]) ** 2 + (y - t[1]) ** 2) / (2 * t[2] ** 2))


def err(t, x, y, z):
    return fun(t, x, y) - z


p2 = least_squares(err, p0, args=(x3, y3, zr))
print("p2:", p2)
p3 = leastsq(err, p0, args=(x3, y3, zr))
print("p3:", p3)
fig.savefig("data/7_15.png", dpi=500)
plt.show()

## 5.17  含未知分段点的分段线性拟合

13 对 (x0, y0) 数据，模型在分界点 k 两侧分别为 $a+bx$（$x<k$）与 $c+dx$（$x\geq k$），用布尔因子 `(x < k)`、`(x >= k)` 拼接成单一函数。`curve_fit(..., bounds=(LB, UB))`：前 4 个参数上下界为 $\mp\infty$，k 限定在 $[\min(x_0), \max(x_0)]$（取 ndarray 的 `.min()/.max()`），打印拟合参数；在 `np.linspace(x0.min(), x0.max(), 100)` 上画已知数据的 `'*'` 散点与拟合函数曲线，图例 data/fitted（源程序 ex7_17.py）。

In [ ]:
x0 = np.array(
    [0.81, 0.91, 0.13, 0.91, 0.63, 0.098, 0.28, 0.55, 0.96, 0.96, 0.16, 0.97, 0.96]
)
y0 = np.array(
    [
        0.17,
        0.12,
        0.16,
        0.0035,
        0.37,
        0.082,
        0.34,
        0.56,
        0.15,
        -0.046,
        0.17,
        -0.091,
        -0.071,
    ]
)


def y(x, a, b, c, d, k):
    return (a + b * x) * (x < k) + (c + d * x) * (x >= k)


LB = [-np.inf] * 4
LB.append(x0.min())
UB = [np.inf] * 4
UB.append(x0.max())
p = curve_fit(y, x0, y0, bounds=(LB, UB))[0]
print("拟合参数为:", p)
x = np.linspace(x0.min(), x0.max(), 100)
fig, ax = plt.subplots()
ax.plot(x0, y0, "*", label="data")
ax.plot(x, y(x, *p), label="fitted")
ax.legend()
plt.show()

## 5.19  三次样条求总排沙量与两阶段多项式拟合

读 `data/7_19.txt`（相邻两行分别为水流量与含沙量），`a[::2, :]`、`a[1::2, :]` 取出并按顺序展平成 liu 与 sha，乘积 y=sha·liu 为排沙量；时刻 $t=(12i-4)\cdot 3600$ 秒（i=1…24）。`interp1d(t, y, "cubic")` 作三次样条插值，在 $[t_1, t_2]$ 取 10000 个节点后用 `np.trapezoid` 数值积分得总含沙量 TL（打印）。左右子图分别画第一阶段前 11 个点与第二阶段其余点的 `'*'` 散点，x 轴标题“第一阶段”“第二阶段”。随后对两阶段分别 `np.polyfit` 拟合 1、2 次多项式，预测值 `np.polyval` 与观测的误差平方和（内置 sum 累加）除以自由度 $n-i$ 开方得剩余标准差 rmse，打印各次系数与 rmse 数组（源程序 ex7_19.py）。

In [ ]:
a = np.loadtxt("data/7_19.txt")
liu = a[::2, :].flatten()
sha = a[1::2, :].flatten()
y = sha * liu
i = np.arange(1, 25)
t = (12 * i - 4) * 3600
t1 = t[0]
t2 = t[-1]
f = interp1d(t, y, "cubic")
tt = np.linspace(t1, t2, 10000)
TL = np.trapezoid(f(tt), tt)
print("总含沙量为：", TL)

fig, (ax, ax2) = plt.subplots(1, 2)
ax.plot(liu[:11], y[:11], "*")
ax.set(xlabel="第一阶段")
ax2.plot(liu[11:], y[11:], "*")
ax2.set(xlabel="第二阶段")

rmse1 = np.zeros(2)
rmse2 = np.zeros(2)
for i in range(1, 3):
    nh1 = np.polyfit(liu[:11], y[:11], i)
    print("第一阶段，", i, "次多项式系数:", nh1)
    yh1 = np.polyval(nh1, liu[:11])
    cha1 = sum((y[:11] - yh1) ** 2)
    rmse1[i - 1] = np.sqrt(cha1 / (10 - i))
print("剩余标准差分别为：", rmse1)
for i in range(1, 3):
    nh2 = np.polyfit(liu[11:], y[11:], i)
    print("第二阶段，", i, "次多项式系数:", nh2)
    yh2 = np.polyval(nh2, liu[11:])
    cha2 = sum((y[11:] - yh2) ** 2)
    rmse2[i - 1] = np.sqrt(cha2 / (12 - i))
print("剩余标准差分别为：", rmse2)
plt.show()